In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce_orders
(
order_id INT,
customer_name STRING,
city STRING,
product STRING,
quantity INT,
price INT,
order_status STRING
)
USING DELTA;

In [0]:
%sql
INSERT INTO ecommerce_orders VALUES
(101,'Amit Sharma','Hyderabad','Laptop',1,75000,'Placed'),
(102,'Priya Reddy','Bangalore','Mobile',2,30000,'Placed'),
(103,'Rohit Mehta','Mumbai','Headphones',3,2000,'Shipped'),
(104,'Sneha Iyer','Chennai','Laptop',1,72000,'Placed'),
(105,'Karan Patel','Ahmedabad','Tablet',2,25000,'Cancelled'),
(106,'Ananya Das','Kolkata','Mobile',1,28000,'Placed'),
(107,'Vikram Singh','Delhi','Camera',1,55000,'Placed'),
(108,'Meera Nair','Bangalore','Laptop',1,80000,'Placed');

num_affected_rows,num_inserted_rows
8,8


In [0]:
%sql
INSERT INTO ecommerce_orders VALUES (109, 'Rahul Verma', 'Mumbai', 'Tablet', 1, 27000, 'Placed');

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
INSERT INTO ecommerce_orders (order_id, customer_name, product) VALUES 
(110, 'Arjun Nair', 'Mobile'), 
(111, 'Kavya Reddy', 'Laptop');

num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql
INSERT INTO ecommerce_orders VALUES (112, 'Suresh', 'Chennai', 'Mouse', 1, 500, 'Shipped');

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
INSERT INTO ecommerce_orders VALUES (113, 'Vijay', 'Delhi', 'Headphones', 5, 2000, 'Placed');

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
INSERT INTO ecommerce_orders VALUES 
(114, 'Sai', 'Hyderabad', 'Mobile', 1, 15000, 'Placed'),
(115, 'Ram', 'Hyderabad', 'Laptop', 1, 60000, 'Placed'),
(116, 'Ali', 'Hyderabad', 'Tablet', 1, 20000, 'Placed');

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
UPDATE ecommerce_orders SET order_status = 'Shipped' WHERE order_id = 101;

num_affected_rows
1


In [0]:
%sql
UPDATE ecommerce_orders SET quantity = quantity + 1 WHERE order_id = 102;

num_affected_rows
1


In [0]:
%sql
UPDATE ecommerce_orders SET price = 78000 WHERE product = 'Laptop';

num_affected_rows
5


In [0]:
%sql
UPDATE ecommerce_orders SET city = 'Secunderabad' WHERE customer_name = 'Amit Sharma';

num_affected_rows
1


In [0]:
%sql
UPDATE ecommerce_orders SET order_status = 'Delivered' WHERE product = 'Mobile';

num_affected_rows
4


In [0]:
%sql
UPDATE ecommerce_orders SET price = 2500 WHERE product = 'Headphones';

num_affected_rows
2


In [0]:
%sql
UPDATE ecommerce_orders SET price = price + 1000 WHERE product = 'Tablet';

num_affected_rows
3


In [0]:
%sql
UPDATE ecommerce_orders SET order_status = 'Processing' WHERE city = 'Bangalore';

num_affected_rows
2


In [0]:
%sql
UPDATE ecommerce_orders SET quantity = 2 WHERE quantity = 1;

num_affected_rows
10


In [0]:
%sql
UPDATE ecommerce_orders SET city = 'Surat' WHERE city = 'Ahmedabad';

num_affected_rows
1


In [0]:
%sql
DELETE FROM ecommerce_orders WHERE order_status = 'Cancelled';

num_affected_rows
1


In [0]:
%sql
DELETE FROM ecommerce_orders WHERE quantity > 3;

num_affected_rows
0


In [0]:
%sql
DELETE FROM ecommerce_orders WHERE product = 'Camera';
DELETE FROM ecommerce_orders WHERE city = 'Kolkata';
DELETE FROM ecommerce_orders WHERE price < 5000;

num_affected_rows
2


In [0]:
%sql
DELETE FROM ecommerce_orders WHERE customer_name LIKE 'A%';
DELETE FROM ecommerce_orders WHERE product = 'Tablet';
DELETE FROM ecommerce_orders WHERE city = 'Mumbai' AND quantity = 1;
DELETE FROM ecommerce_orders WHERE price > 80000;

num_affected_rows
0


In [0]:
%sql
DELETE FROM ecommerce_orders WHERE order_id = (SELECT MAX(order_id) FROM ecommerce_orders);

num_affected_rows
1


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW incoming_orders AS
SELECT * FROM VALUES
(102,'Priya Reddy','Bangalore','Mobile',4,30000,'Delivered'),
(104,'Sneha Iyer','Chennai','Laptop',1,72000,'Shipped'),
(109,'Rahul Verma','Mumbai','Tablet',2,26000,'Placed'),
(110,'Divya Menon','Kochi','Mobile',1,27000,'Placed'),
(111,'Farhan Ali','Hyderabad','Laptop',1,79000,'Placed')
AS incoming_orders(order_id,customer_name,city,product,quantity,price,order_status);

In [0]:
%sql
MERGE INTO ecommerce_orders AS target
USING incoming_orders AS source
ON target.order_id = source.order_id
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,3,0,2


In [0]:
%sql
SELECT * FROM ecommerce_orders ORDER BY order_id;

order_id,customer_name,city,product,quantity,price,order_status
102,Priya Reddy,Bangalore,Mobile,4,30000,Delivered
104,Sneha Iyer,Chennai,Laptop,1,72000,Shipped
108,Meera Nair,Bangalore,Laptop,2,78000,Processing
109,Rahul Verma,Mumbai,Tablet,2,26000,Placed
110,Divya Menon,Kochi,Mobile,1,27000,Placed
111,Farhan Ali,Hyderabad,Laptop,1,79000,Placed
114,Sai,Hyderabad,Mobile,2,15000,Delivered


In [0]:
%sql MERGE INTO ecommerce_orders AS target
USING incoming_orders AS source
ON target.order_id = source.order_id
WHEN MATCHED AND source.quantity > target.quantity THEN
  UPDATE SET target.quantity = source.quantity;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW daily_updates AS
SELECT * FROM VALUES
(103,'Rohit Mehta','Mumbai','Headphones',5,2200,'Delivered'),
(106,'Ananya Das','Kolkata','Mobile',2,28000,'Shipped'),
(112,'Sanjay Gupta','Delhi','Tablet',1,24000,'Placed')
AS daily_updates(order_id,customer_name,city,product,quantity,price,order_status)

In [0]:
%sql
MERGE INTO ecommerce_orders AS target
USING daily_updates AS source
ON target.order_id = source.order_id
WHEN MATCHED THEN
  UPDATE SET target.quantity = source.quantity, target.order_status = source.order_status
WHEN NOT MATCHED THEN
  INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
3,0,0,3


In [0]:
%sql
SELECT * FROM ecommerce_orders ORDER BY price DESC;

order_id,customer_name,city,product,quantity,price,order_status
111,Farhan Ali,Hyderabad,Laptop,1,79000,Placed
108,Meera Nair,Bangalore,Laptop,2,78000,Processing
104,Sneha Iyer,Chennai,Laptop,1,72000,Shipped
102,Priya Reddy,Bangalore,Mobile,4,30000,Delivered
106,Ananya Das,Kolkata,Mobile,2,28000,Shipped
110,Divya Menon,Kochi,Mobile,1,27000,Placed
109,Rahul Verma,Mumbai,Tablet,2,26000,Placed
112,Sanjay Gupta,Delhi,Tablet,1,24000,Placed
114,Sai,Hyderabad,Mobile,2,15000,Delivered
103,Rohit Mehta,Mumbai,Headphones,5,2200,Delivered
